# DLE602 Week 6 Activity Notebook  
## Linear Factor Models: Learning Better Representations of Data

In this activity, you will explore why linear factor models are useful in machine learning and deep learning. The focus is on intuition first, followed by simple Python demonstrations.

You will work with:

- Principal Component Analysis (PCA)
- Independent Component Analysis (ICA)
- Slow Feature Analysis (SFA) intuition
- Sparse Coding intuition

The goal is not to memorise equations. The goal is to understand how these techniques help simplify data and learn better representations.

## Learning Outcomes

By the end of this notebook, you should be able to:

1. Explain why dimensionality reduction is useful.
2. Describe the difference between PCA, ICA, SFA and Sparse Coding.
3. Use PCA to reduce data dimensions and interpret explained variance.
4. Explain the cocktail party problem using ICA.
5. Connect linear factor models to deep learning workflows.

## Setup

Run the cell below first. If a package is missing, install it using:

```python
pip install numpy matplotlib scikit-learn
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA, FastICA, SparseCoder
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Setup complete. You are ready to begin Week 6 activities.")

# Part 1: Why Do We Need PCA?

Real-world datasets often contain many variables. Some variables may be highly related to each other. PCA helps reduce the number of features while preserving most of the important information.

### Stop and Think

If a dataset has 100 features, do you think all 100 features are always equally useful? Why or why not?

In [ ]:
# Generate simple correlated 2D data
n_samples = 250
x = np.random.normal(0, 1, n_samples)
y = 0.7 * x + np.random.normal(0, 0.35, n_samples)
X = np.column_stack((x, y))

# Standardise data before PCA
X_scaled = StandardScaler().fit_transform(X)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
components = pca.components_

plt.figure(figsize=(7, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], alpha=0.7)
origin = np.mean(X_scaled, axis=0)

# Draw principal component directions
for i, comp in enumerate(components):
    plt.arrow(origin[0], origin[1], comp[0]*2, comp[1]*2,
              head_width=0.08, linewidth=2, length_includes_head=True)
    plt.text(comp[0]*2.2, comp[1]*2.2, f"PC{i+1}", fontsize=12)

plt.title("PCA finds directions of maximum variation")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True)
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by PC1:", round(pca.explained_variance_ratio_[0] * 100, 2), "%")

### What to Observe

- PC1 points in the direction where the data varies the most.
- PC2 captures the remaining variation.
- If PC1 explains most of the variance, the 2D data can be represented reasonably well using only 1 dimension.

### Try It Yourself

Change the noise level in this line:

```python
y = 0.7 * x + np.random.normal(0, 0.35, n_samples)
```

Try values such as `0.1`, `0.7`, or `1.2`. What happens to the explained variance?

# Part 2: PCA for High-Dimensional Data

Now we simulate a dataset with 12 features and reduce it to 2 principal components. This is similar to reducing a complex dataset into a simpler representation for visualisation or modelling.

In [ ]:
# Create high-dimensional data
X_hd, labels = make_blobs(
    n_samples=400,
    centers=4,
    n_features=12,
    cluster_std=2.5,
    random_state=42
)

X_hd_scaled = StandardScaler().fit_transform(X_hd)

# Reduce from 12 features to 2 principal components
pca_2 = PCA(n_components=2)
X_2d = pca_2.fit_transform(X_hd_scaled)

plt.figure(figsize=(7, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, alpha=0.75)
plt.title("High-dimensional data reduced to 2 PCA components")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)
plt.show()

print("Original number of features:", X_hd.shape[1])
print("Reduced number of features:", X_2d.shape[1])
print("Variance explained by 2 components:", round(sum(pca_2.explained_variance_ratio_) * 100, 2), "%")

### Explained Variance

Explained variance tells us how much information is retained by the selected principal components.

A higher value means more information is preserved.

In [ ]:
# Compare explained variance for different numbers of components
pca_full = PCA().fit(X_hd_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o')
plt.axhline(y=0.90, linestyle='--', label='90% variance threshold')
plt.title("Cumulative explained variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.xticks(range(1, len(cumulative_variance)+1))
plt.grid(True)
plt.legend()
plt.show()

for i, value in enumerate(cumulative_variance, start=1):
    print(f"{i} component(s): {round(value*100, 2)}% variance retained")

### Stop and Think

If 5 components retain 90% of the information, would you still use all 12 original features? Explain your reasoning.

# Part 3: ICA and the Cocktail Party Problem

ICA is useful when signals are mixed together and we want to recover the independent sources.

A common example is the **cocktail party effect**:

Many voices → mixed sound → focus on one speaker → separate useful signal from noise.

In [ ]:
# Create two original signals
n = 1000
time = np.linspace(0, 8, n)

source_1 = np.sin(2 * time)                # smooth wave
source_2 = np.sign(np.sin(3 * time))       # square-like wave
S = np.c_[source_1, source_2]
S += 0.05 * np.random.normal(size=S.shape)
S /= S.std(axis=0)

# Mix the signals
mixing_matrix = np.array([[1, 0.5], [0.4, 1]])
X_mixed = np.dot(S, mixing_matrix.T)

# Recover signals using ICA
ica = FastICA(n_components=2, random_state=42, whiten='unit-variance')
S_recovered = ica.fit_transform(X_mixed)

plt.figure(figsize=(10, 4))
plt.plot(time, S[:, 0], label='Original Source 1')
plt.plot(time, S[:, 1], label='Original Source 2', alpha=0.8)
plt.title("Original independent sources")
plt.xlabel("Time")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(time, X_mixed[:, 0], label='Mixed Signal 1')
plt.plot(time, X_mixed[:, 1], label='Mixed Signal 2', alpha=0.8)
plt.title("Mixed signals observed by sensors")
plt.xlabel("Time")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(time, S_recovered[:, 0], label='Recovered Component 1')
plt.plot(time, S_recovered[:, 1], label='Recovered Component 2', alpha=0.8)
plt.title("Signals recovered using ICA")
plt.xlabel("Time")
plt.legend()
plt.grid(True)
plt.show()

### What to Observe

- The mixed signals look different from the original sources.
- ICA attempts to recover the independent components from the mixed signals.
- This is different from PCA because ICA focuses on independence, not maximum variance.

### Reflection Question

When would ICA be more suitable than PCA?

# Part 4: Slow Feature Analysis (SFA) Intuition

SFA is useful when data changes over time, but we want to identify features that remain stable or change slowly.

Example applications:

- Robot vision
- Video analysis
- Object tracking
- Autonomous driving

In [ ]:
# Simple SFA intuition: fast changing signal and slow changing trend
n = 500
time = np.linspace(0, 20, n)
fast_signal = np.sin(8 * time) + 0.3 * np.random.normal(size=n)
slow_feature = np.sin(0.5 * time)
combined_signal = fast_signal + slow_feature

plt.figure(figsize=(10, 4))
plt.plot(time, combined_signal, label='Rapidly changing observed signal')
plt.plot(time, slow_feature, linewidth=3, label='Slow underlying feature')
plt.title("SFA intuition: extracting slowly changing structure")
plt.xlabel("Time")
plt.legend()
plt.grid(True)
plt.show()

### Stop and Think

In a video, every frame changes quickly. However, the identity of an object may remain stable. Why might this be useful for deep learning?

# Part 5: Sparse Coding Intuition

Sparse Coding represents data using only a small number of active features.

In simple words:

Many possible features → only a few become active → efficient representation.

In [ ]:
# Demonstrate sparse activations
n_features = 30
n_active = 5
activations = np.zeros(n_features)
active_indices = np.random.choice(range(n_features), n_active, replace=False)
activations[active_indices] = np.random.uniform(0.5, 1.0, n_active)

plt.figure(figsize=(9, 4))
plt.bar(range(1, n_features + 1), activations)
plt.title("Sparse Coding intuition: only a few features are active")
plt.xlabel("Feature index")
plt.ylabel("Activation strength")
plt.grid(True)
plt.show()

print("Number of available features:", n_features)
print("Number of active features:", n_active)
print("Active feature indices:", sorted(active_indices + 1))

### What to Observe

Sparse Coding does not activate every possible feature. It uses only a small number of meaningful features.

This is useful in:

- Image compression
- Efficient representation learning
- Neuroscience-inspired deep learning

# Final Comparison

| Technique | Main Question | Best Used When |
|---|---|---|
| PCA | Can we reduce features while keeping most information? | Too many correlated features |
| ICA | Can we separate mixed independent signals? | Audio, EEG, signal separation |
| SFA | Can we find stable features over time? | Video, robotics, time-series |
| Sparse Coding | Can we represent data using only a few active features? | Efficient representation and compression |

# Mini Challenge

Choose one application and identify the most suitable linear factor model.

1. Face recognition with 10,000 pixel values  
2. Separating two speakers in a noisy room  
3. Robot identifying the same object across video frames  
4. Compressing images using only a few important features  

Write your answer below in your own words.

## Student Reflection

Complete the following sentences:

1. Before this activity, I thought dimensionality reduction was...
2. Now I understand that PCA is useful because...
3. ICA is different from PCA because...
4. One possible use of linear factor models in my Assessment 2 project is...